In [ ]:
# %pip install pydantic

In [ ]:
import pandas as pd
from loguru import logger
from pydantic import (
    BaseModel,
    ValidationError,
    ValidationInfo,
    field_validator,
)

In [ ]:
error_message: dict = {
    "check_mandatory": "[{}][Check mandatory] Required field.",
    "check_data_type": "[{}][Check data type] Data type must be '{}'.",
    "check_in_range_numeric": "[{}][Check in range numeric] Value must be in range [{}, {}].",
    "check_correct_datetime_format": "[{}][Check correct datetime format] Value must be correct format: '{}'.",
    "check_in_range_datetime": "[{}][Check in range datetime] Value must be in range [{}, {}].",
    "check_in_range_string_length": "[{}][Check in range string length] Data length must be in range [{}, {}].",
    "check_unique": "[{}][Check unique] Value is duplicated.",
    "check_enum": "[{}][Check enum] Value must be in enum: {}."
}

In [ ]:
@logger.catch
def mapping_to_snake_case(column_name: str) -> str:
    return column_name.lower().strip().replace(" ", "_").replace("/", "_")

In [ ]:
master_data_setup_file_path: str = "/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/prevalidation_data_2503/Data Template 03_Master Data Setup_SLP3P4 1.xlsx"

In [ ]:
sheet_names = pd.ExcelFile(master_data_setup_file_path).sheet_names
# sheet_names

In [ ]:
df_suspension_reason: pd.DataFrame = pd.read_excel(master_data_setup_file_path, sheet_name="SuspensionReason")
df_suspension_reason.head(2)

In [ ]:
suspension_reason_origin_columns: list = df_suspension_reason.columns.tolist()
df_suspension_reason.columns = df_suspension_reason.columns.map(mapping_to_snake_case)
df_suspension_reason.head(2)

In [ ]:
@logger.catch
def check_datetime_format(value: str, column_name: str, datetime_format: str = "%d/%m/%Y %H:%M:%S") -> str:
    try:
        datetime.datetime.strptime(value, datetime_format)
    except ValueError:
        print(error_message["check_correct_datetime_format"].format(column_name, datetime_format))

In [ ]:
@logger.catch
def check_date_time_logic(created_on: str, modified_on: str, datetime_format: str = "%d/%m/%Y %H:%M:%S") -> str:
        created_on = datetime.datetime.strptime(created_on, datetime_format)
        modified_on = datetime.datetime.strptime(modified_on, datetime_format)

        if created_on and modified_on:
            if created_on > modified_on:
                raise ValueError("Created on must be before modified on")

In [ ]:
from enum import Enum
from typing import Literal, List, Optional, Union, Any
import datetime
import numpy as np
from pydantic import Field, model_validator
from typing_extensions import Self


class SuspensionReasonValidation(BaseModel):
    suspension_reason_name: str = Field(max_length=50, min_length=3)
    description: Any = Field(max_length=200, min_length=0)
    status: Literal["Active", "Inactive"]
    created_on: Optional[str] = Field(None)
    created_by: Optional[str] = Field(None)
    modified_on: Optional[str] = Field(None)
    modified_by: Optional[str] = Field(None)

    @field_validator("created_on", "modified_on")
    @classmethod
    def check_datetime_format(cls, value: Optional[str], info: ValidationInfo) -> Optional[str]:
        if value:
            check_datetime_format(value, info.field_name, "%d/%m/%Y %H:%M:%S")
        return value

    @model_validator(mode="after")
    def check_date_time_logic(model: Self) -> Self:
        value = model.modified_on
        if value:
            created_on = model.created_on
            check_date_time_logic(created_on, value, "%d/%m/%Y %H:%M:%S")
            return model

df_suspension_reason["description"] = df_suspension_reason["description"].astype(str)
suspension_reason_data: list = df_suspension_reason.to_dict(orient="records")
results = []

for data in suspension_reason_data:
    try:
        obj = SuspensionReasonValidation(**data)
        results.append(obj)

    except ValidationError as e:
        print(e.errors())

## InternalSuspendedList

In [ ]:
df_internal_suspended_list: pd.DataFrame = pd.read_excel(master_data_setup_file_path, sheet_name="InternalSuspendedList")
df_internal_suspended_list.head(2)

In [ ]:
origin_internal_suspended_list_columns: list = df_internal_suspended_list.columns.tolist()
df_internal_suspended_list.columns = df_internal_suspended_list.columns.map(mapping_to_snake_case)
df_internal_suspended_list.head(2)

In [ ]:
suspended_reason_name: List = df_suspension_reason["suspension_reason_name"].tolist()

In [ ]:
class InternalSuspendedListValidation(BaseModel):
    id: str = Field(max_length=50, min_length=1)
    debtor_type: Literal["Individual", "Company"]
    nric_fin: Optional[str] = Field(None)
    uen: Optional[str] = Field(None)
    effective_from: Optional[str]
    effective_to: Optional[str] = Field(None)
    bad_debt_write_off_amount: Union[float, int] = Field(None)
    suspended_reason: Literal[suspended_reason_name]
    remarks: Optional[str]
    status: Literal["Active", "Inactive"]
    created_on: Optional[str] = Field(None)
    created_by: Optional[str] = Field(None)
    modified_on: Optional[str] = Field(None)
    modified_by: Optional[str] = Field(None)

    @field_validator("effective_from", "effective_to", "created_on", "modified_on")
    @classmethod
    def check_datetime_format(cls, value: Optional[str], info: ValidationInfo) -> Optional[str]:
        # get column name from validator
        field_name = info.field_name
        if value:
            if field_name == "effective_from" or field_name == "effective_to":
                check_datetime_format(value, field_name, "%d/%m/%Y")
            else:
                check_datetime_format(value, field_name, "%d/%m/%Y %H:%M:%S")
        return value

    @model_validator(mode="after")
    def check_nric_fin_logic(model: Self) -> Self:
        nric_fin = model.nric_fin
        uen = model.uen
        debtor_type = model.debtor_type
        if debtor_type == "Individual" and not nric_fin:
            raise ValueError("Nric fin is required for individual debtor type")
        if debtor_type == "Company" and not uen:
            raise ValueError("Uen is required for company debtor type")
        return model

    @model_validator(mode="after")
    def check_date_time_logic(model: Self) -> Self:
        value = model.modified_on
        if value:
            created_on = model.created_on
            check_date_time_logic(created_on, value, "%d/%m/%Y %H:%M:%S")
            return model
# fill na for float, int
# fill "" for string
df_internal_suspended_list = df_internal_suspended_list.convert_dtypes()
numeric_columns = df_internal_suspended_list.select_dtypes(include=[np.number]).columns.tolist()
df_internal_suspended_list[numeric_columns] = df_internal_suspended_list[numeric_columns].fillna(np.nan)

string_columns = df_internal_suspended_list.select_dtypes(include=[object]).columns.tolist()
df_internal_suspended_list[string_columns] = df_internal_suspended_list[string_columns].fillna("")

df_internal_suspended_list["remarks"] = df_internal_suspended_list["remarks"].astype(str)
df_internal_suspended_list["status"] = df_internal_suspended_list["status"].astype(str).map(lambda x: x.strip().title())
df_internal_suspended_list["debtor_type"] = df_internal_suspended_list["debtor_type"].astype(str).map(lambda x: x.strip().title())
internal_suspended_list_data: list = df_internal_suspended_list.to_dict(orient="records")
results = []

for data in internal_suspended_list_data:
    try:
        obj = InternalSuspendedListValidation(**data)
        results.append(obj)

    except ValidationError as e:
        print(e.errors())
## InternalSuspendedList
internal_suspended_list_data: list = df_internal_suspended_list.to_dict(orient="records")
results = []

for data in internal_suspended_list_data:
    try:
        obj = InternalSuspendedListValidation(**data)
        results.append(obj)

    except ValidationError as e:
        print(e.errors())